In [5]:
# %% 
# ================================================================
# 8회차 통합 실습: 가설검정 전체 구조 이해하기
# ================================================================
# 이 실습의 목표는 다음입니다.
# 1. 표본 평균이 어떻게 흔들리는지 확인한다.
# 2. 검정통계량(t값)을 직접 계산해본다.
# 3. p-value를 계산해본다.
# 4. 1종 오류를 시뮬레이션으로 확인한다.
# 5. 표본 크기가 검정력에 어떤 영향을 주는지 확인한다.
# 6. 효과 크기를 계산해본다.
#
# 이 실습은 "계산 연습"이 아니라
# "판단 구조 이해"가 목적입니다.
# ================================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)  # 재현성을 위한 시드 고정


In [6]:
# %%
# ================================================================
# 1단계: 가설 설정
# ================================================================
# 상황 설정:
# 어떤 집단의 모평균이 100이라고 주장(H0)합니다.
# 우리는 표본을 뽑아서 이 주장을 검정하려고 합니다.

mu_0 = 100      # 귀무가설의 평균
sigma = 15      # 모집단 표준편차 (알고 있다고 가정)
n = 30          # 표본 크기

# 실제 모집단 평균 (여기서는 일부러 차이를 만들어 보겠습니다)
true_mu = 105   # 실제 평균 (H0와 다름)

# 표본 생성
sample = np.random.normal(true_mu, sigma, n)


In [7]:

# %%
# ================================================================
# 2단계: 표본 통계량 계산
# ================================================================

sample_mean = np.mean(sample)
sample_std = np.std(sample, ddof=1)  # 표본 표준편차 (자유도 1 보정)
SE = sample_std / np.sqrt(n)         # 표준오차

print("표본 평균:", round(sample_mean, 3))
print("표본 표준편차:", round(sample_std, 3))
print("표준오차(SE):", round(SE, 3))

# 질문:
# 왜 표준편차가 아니라 표준오차를 쓰는가?
# -> 우리는 '개별 데이터의 흔들림'이 아니라
#    '평균의 흔들림'을 알고 싶기 때문.


표본 평균: 102.178
표본 표준편차: 13.5
표준오차(SE): 2.465


In [8]:


# %%
# ================================================================
# 3단계: 검정통계량(t값) 계산
# ================================================================
# 공식:
# t = (표본평균 - 가설평균) / SE

t_stat = (sample_mean - mu_0) / SE
df = n - 1  # 자유도

print("t 통계량:", round(t_stat, 3))
print("자유도:", df)

# 해석:
# t값은 "차이 / 불확실성"입니다.
# 값이 클수록 귀무가설에서 멀어진 것입니다.


t 통계량: 0.884
자유도: 29


In [9]:


# %%
# ================================================================
# 4단계: p-value 계산 (양측검정)
# ================================================================

p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=df))

print("p-value:", round(p_value, 5))

alpha = 0.05

if p_value < alpha:
    print("결론: 유의수준 5%에서 귀무가설을 기각한다.")
else:
    print("결론: 귀무가설을 기각하지 않는다.")

# 질문:
# p-value는 무엇의 확률인가?
# -> H0가 참이라고 가정했을 때,
#    지금과 같거나 더 극단적인 값이 나올 확률.


p-value: 0.38419
결론: 귀무가설을 기각하지 않는다.


In [10]:


# %%
# ================================================================
# 5단계: scipy로 t-test 실행 (검증용)
# ================================================================

t_test_result = stats.ttest_1samp(sample, mu_0)

print("scipy t값:", round(t_test_result.statistic, 3))
print("scipy p-value:", round(t_test_result.pvalue, 5))

# 직접 계산한 값과 비교해보세요.
# 거의 동일해야 합니다.



scipy t값: 0.884
scipy p-value: 0.38419


In [11]:

# %%
# ================================================================
# 6단계: 1종 오류 시뮬레이션
# ================================================================
# 이제 H0가 '정말로 참'인 상황을 가정합니다.
# true_mu = mu_0 로 설정합니다.
# 그리고 여러 번 반복 실험을 해서
# 실제로 약 5% 정도가 잘못 기각되는지 확인합니다.

true_mu = mu_0
n_sim = 5000
rejections = 0

for _ in range(n_sim):
    sim_sample = np.random.normal(true_mu, sigma, n)
    t_stat = (np.mean(sim_sample) - mu_0) / (np.std(sim_sample, ddof=1)/np.sqrt(n))
    p_val = 2 * (1 - stats.t.cdf(abs(t_stat), df=n-1))
    
    if p_val < alpha:
        rejections += 1

print("1종 오류 비율 (이론상 0.05):", rejections / n_sim)

# 관찰:
# 0.05 근처 값이 나올 것입니다.
# 이것이 유의수준의 장기적 의미입니다.



1종 오류 비율 (이론상 0.05): 0.0558


In [12]:

# %%
# ================================================================
# 7단계: 표본 크기와 검정력
# ================================================================
# 이제 실제로 차이가 있는 경우(true_mu = 105)
# 표본 크기를 키우면 검정력이 어떻게 변하는지 봅니다.

true_mu = 105
sample_sizes = [20, 50, 100, 200]
power_results = []

for n in sample_sizes:
    rejections = 0
    for _ in range(2000):
        sim_sample = np.random.normal(true_mu, sigma, n)
        t_stat = (np.mean(sim_sample) - mu_0) / (np.std(sim_sample, ddof=1)/np.sqrt(n))
        p_val = 2 * (1 - stats.t.cdf(abs(t_stat), df=n-1))
        
        if p_val < alpha:
            rejections += 1
    
    power = rejections / 2000
    power_results.append(power)

for n, p in zip(sample_sizes, power_results):
    print(f"표본크기 {n} -> 검정력: {round(p,3)}")

# 관찰:
# 표본 크기가 커질수록 검정력이 증가합니다.


표본크기 20 -> 검정력: 0.293
표본크기 50 -> 검정력: 0.635
표본크기 100 -> 검정력: 0.906
표본크기 200 -> 검정력: 0.995


In [13]:


# %%
# ================================================================
# 8단계: 효과 크기 (Cohen's d)
# ================================================================
# 효과 크기는 차이의 '상대적 크기'입니다.

true_mu = 105
n = 30
sample = np.random.normal(true_mu, sigma, n)

mean_diff = np.mean(sample) - mu_0
pooled_std = np.std(sample, ddof=1)

cohens_d = mean_diff / pooled_std

print("Cohen's d:", round(cohens_d, 3))

# 해석 기준:
# 0.2 -> 작은 효과
# 0.5 -> 중간 효과
# 0.8 -> 큰 효과

# 질문:
# p-value는 작지만 d가 작다면?
# -> 통계적으로 유의하지만 실질적으로는 작은 효과일 수 있습니다.

Cohen's d: -0.081
